# Day 29 — GRPO Intuition (vs. PPO / DPO)

> **Deliverable:** GRPO intuition notes. What the reward actually is, and what "group relative" is really doing.
> The point of these notes isn't "how to get GRPO running." It's *why it's built this way, and what that design costs you.*

---

## 1. Where does the reward come from? Two very different routes

Picture yourself grading a student's writing. You've got two completely different ways to score it.

**Route one: check the answer key (rule-based reward)**

You hand them a math problem — "3 × 24 = ?" — and the answer is 72. There's nothing to *feel out* here. A few lines of code can grade it: output 72, get 1 point; anything else, 0. No gray area, no model needed to pass judgment. One line does it:

```python
reward = 1 if answer == 72 else 0
```

**Route two: hire a judge (reward model / preference)**

Now you hand them "write a short essay about friendship." There *is* no answer key. No program can tell you whether it's any good. So what now?

You train a judge model ahead of time — feed it a pile of "humans preferred this one over that one" data until it learns to imitate human taste. Once training is done, the judge gets **frozen** (no more updates). From then on, every time you need a score, you hand the essay to the judge and it spits out a number, say 0.8.

Side by side:

| | Rule-based reward | Reward model (preference) |
|---|---|---|
| Who grades | A hardcoded rule (is the answer right, does the code run) | A trained, **frozen** judge model |
| Nature of the score | Objective, verifiable — anyone computing it gets the same number | Subjective, imitating human taste — swap the judge and the scores shift |
| Poster child | DeepSeekMath / DeepSeek-R1 (GRPO) | InstructGPT (PPO) |
| Extra cost | Basically zero — one conditional | You have to spend time and money training a whole reward model first |

**DeepSeekMath takes route one.** The model spits out a long chain of "reasoning + final answer," and the system only looks at **that last number**. Is it 72? The whole output gets 1 point. Isn't it? Zero.

This is where I got stuck: **there's no answer key for the middle.** The model might have taken five totally different routes to reach 72, and nobody ever told it "here's how step three should look." So how does that single point get distributed across all the tokens in between?

The answer is **broadcast**. Since the output is one unit, we just assume every token along that path shares the credit (or the blame), and we **copy the same number onto every single token position**. Not weighted by attention, not weighted by importance. Same number, everywhere.

> ⚠️ **Don't get the order backwards.** What gets broadcast is **Â_i from §2** (the +0.577 / −1.732 numbers), *not* the raw r_i (1 or 0).
> The sequence is: compute mean/std across the whole group **first** → get Â_i for each output → **then** broadcast that Â_i across every token in that output.

This is exactly why GRPO doesn't need per-token supervision. You bluntly copy the sequence-level verdict onto every position and call it a day.

---

## 2. So what does "group relative" actually mean?

**TL;DR:** I don't need a teacher standing next to me predicting my grade. I just compare myself against my cohort. If I score above the group average, I must have done *something* right — so the model reinforces it.

Here's the simplest version.

Take "3 × 24 = ?" with G = 4 (sample four answers to the same question — real training usually uses 8–64, but 4 keeps the arithmetic doable by hand). The model produces four different attempts:

| # | What the model wrote | Final answer | Reward r_i |
|---|---|---|---|
| o_1 | 3×24 = 3×20+3×4 = 60+12 = 72 | 72 | 1 |
| o_2 | 3×24 = 3×25−3 = 75−3 = 72 | 72 | 1 |
| o_3 | 3×24 = 3×20+3×4 = 60+8 = 68 | 68 | 0 |
| o_4 | 3×24 = 72 (no work shown, just guessed) | 72 | 1 |

**Step one: compute the group's mean and standard deviation**

- mean = (1+1+0+1) / 4 = 0.75
- std ≈ 0.433

**Step two: compute each output's "advantage" (Â_i)**

Â_i = (r_i − mean) / std

- **o_1:** (1 − 0.75) / 0.433 ≈ **+0.577** → positive, reinforce this path. *(Nice, I did something right — keep going!)*
- **o_2:** same, **+0.577** → reinforce. *(Me too! Keep going!)*
- **o_3:** (0 − 0.75) / 0.433 ≈ **−1.732** → negative, suppress this path. *(Whoa, hold up.)*
- **o_4:** **+0.577** → reinforced anyway. The work is garbage but the answer happened to be right. *(I straight-up fluked it, but a win's a win — keep going!)*

> That o_4 row matters. Remember it. It's what a rule-based reward loophole looks like at toy scale: **right answer, empty reasoning, full marks.** §9 comes back to settle the score with it.

**The thing worth noticing:** that 0.75 baseline **wasn't handed to the model by anyone, and it wasn't guessed by a second model.** It's the actual average score this batch of four earned for itself. PPO needs an extra value model to **guess** "roughly how many points is this question worth?" as its baseline. GRPO just lets the model take the test G times and uses the **real scores** as the baseline. No extra model, no guessing.

That's it — **group** (a batch of samples) plus **relative** (measured against that batch's own average).

---

## 3. Where I got it wrong ①: why PPO's critic is "expensive"

My first instinct was: "because it has to score every single token, and there are tons of tokens, so it must be slow and expensive." **That instinct is wrong**, and it's worth understanding *why* it's wrong.

**First, why a Transformer isn't thinking "one word at a time"**

RNNs (the pre-Transformer architecture) really did process sequentially: you can't compute word 2 until word 1 is done. Like standing in line. But attention computes **the entire sequence in parallel** — that's precisely why Transformers beat RNNs and replaced them.

So "produce a result at every output position" is **almost a free byproduct** in a Transformer. One forward pass, and the hidden state at every position is already sitting there. All the value model has to do is hang a tiny linear layer off each hidden state (squash a few hundred dimensions down to one number). That compute is negligible.

**The actually expensive parts are these two:**

**(1) The value model is an entire separate network, roughly as big as the policy model.**

It's not "one extra layer." It's a full stack of transformer blocks with its own attention and its own FFN, running **its own complete forward + backward pass**. During training, that means parameters, gradients, and optimizer states (plus Adam's momentum buffers) — **a whole second copy of all three**. Take the "how much memory does one model eat" table from the Day 27 QLoRA notes and multiply by two models.

**(2) The supervision signal is brutally sparse.**

The reward model typically hands out one score at the end of the sequence and zeros everywhere else. But the value function's job is to accurately estimate, **at every single token position**, "from here on, how many points can I expect on average?" So: one supervision signal (one number), and you're trying to teach the model to produce accurate estimates at a hundred positions. That's a hundred-fold gap in signal density.

That's a "hard to learn" problem, which is a different problem from "expensive" — but the paper raises them together.

GRPO sidesteps both at once: **don't train a value model at all.** Use the sampled group's own statistics (that mean/std from §2) as the baseline. The entire separate network is gone, and there's no sparse-signal learning problem either — because there's nothing to learn. You just compute it.

▲ See `day27_qlora_memory` for the memory cost breakdown.

---

## 4. Where I got it wrong ②: "comparing against the right answer" doesn't make it supervised learning

Second misconception: "GRPO checks whether the answer is correct at the end — isn't that just supervised learning?" Good question, but no. The distinction isn't **whether there's a correct answer**. It's **who wrote the path that leads to it.**

**How SFT (supervised fine-tuning) does it:**

The teacher gives you the question *and* walks you through the solution. Training data looks like:

> Q: What's 3×24?
> A: 3×24 = 3×20 + 3×4 = 60 + 12 = **72**

The model's job is to **imitate this string of text** — learn to write the same way next time it sees a similar problem. It doesn't need to *figure out* the method, just memorize and reproduce its **style**. The loss reflects that: token by token, compare the model's output distribution against the reference (cross-entropy). **It's learning what the answer looks like** — does it resemble how the teacher wrote it?

Basically teaching to the test.

**How GRPO (RL) does it:**

The teacher gives you the question and the destination — 72 — and nothing else.

> Q: What's 3×24?
> (no worked solution provided)

The model has to generate an entire solution itself. It might be right, might be wrong. If it's wrong (like o_3 above, landing on 68) it gets a 0, and nobody tells it *which step* broke or how to fix it. If it's right, it gets a 1. The model has to grind through **enormous numbers of attempts** — this one problem might get practiced thousands of times — until it discovers, on its own, something like "breaking it up this way makes me less likely to slip." **It's learning the logic**, not memorizing an example. It's developing a method it can actually reuse.

**And that's why 72 isn't a "label" in the supervised-learning sense.** 72 is just the coordinates of the destination. The road there was walked entirely by the model, with nobody holding its hand. That trial-and-error, find-your-own-route process is exactly why RL can produce real reasoning instead of surface-level mimicry.

---

## 5. ★ Where DPO fits

DPO (Direct Preference Optimization) sits between SFT and GRPO. How it works:

You give the model a pair — one good answer, one bad:

> **A (good):** 3×24 = 3×20+3×4 = 60+12 = 72
> **B (bad):** 3×24 = 3×20+3×5 = 60+15 = 75 *(wrong)*

DPO uses a closed-form objective to nudge the parameters directly, so the model becomes more likely to say things like A and less likely to say things like B. No judge model scoring in real time, no sampling G outputs per question. Simpler, cheaper, far more stable to train.

**My first thought was "DPO can't do reasoning" — but whether it learns reasoning depends entirely on whether A contains reasoning.**

- If A in your training data already shows full worked steps, DPO absolutely picks up that reasoning style.
- But DPO has a **structural weakness: it can't explore.** It's forever choosing between the A and B that a human already prepared. It **cannot invent a third solution that neither A nor B contained.**

**Which is where RL (GRPO) wins:**

Go back to §2. With G=4, the model tried four different approaches on its own (o_1 through o_4). One of them might be **a clever method the teacher never demonstrated**. Eureka. As long as that path lands on the right answer, GRPO picks it up and reinforces it. That capacity to discover solutions humans never taught is structurally unavailable to DPO — DPO is boxed into the options a human handed it; GRPO actually lets the model go wander.

This is the core reason DeepSeek insisted on RL (rather than SFT or DPO alone) for DeepSeekMath / R1: if you want reasoning paths that are genuinely the model's own — sometimes better than the human demonstrations — the only way there is to let it crash into walls itself.

---

## 6. ● Where the KL penalty goes: PPO folds it into the reward, GRPO adds it to the loss

§3 covered the value model. There's a second structural difference, and if you only mention the value model when someone asks "how is GRPO different from PPO," you've answered half the question.

**First — why do we need a KL term at all?**

Because you're rewarding the model with a score, and the model will do *anything* to run that score up (that's the reward hacking in §9). The KL term is a leash: *you can change, but you can't drift too far from who you were.*

"Who you were" is the reference model (π_ref) — frozen right after SFT and never touched for the rest of RL.

**PPO's approach: fold KL into the reward**

The per-token reward gets rewritten:

> r_t = (score from the judge) − β · log( π_θ / π_ref )

The further the model drifts from ref at that position, the more gets docked from the reward there. **The corrected r_t is what goes into the advantage computation**, and the "reward including KL" is what the value model has to learn to estimate.

**GRPO's approach: keep the reward clean, put KL on the loss**

The reward stays exactly what the rule gave it — 1 or 0. Untouched. The mean/std in §2 is computed over clean rewards. KL is **computed separately and added onto the loss**.

**Why the placement actually matters** (this is the part that clicked):

§2 does group normalization — subtract mean, divide by std. If KL is baked into the reward, then I'm normalizing "how far this output drifted from ref" along with everything else, and the advantage's meaning gets muddy — it's now a blend of "was this a good answer" *and* "did it wander too far." GRPO keeps them separate:

| | What it governs |
|---|---|
| Advantage | How much better this answer is **than its own group's average** |
| KL term | Keep the whole model **from drifting too far from ref** |

**The estimator is different too**

The naive way to estimate KL is to average log(π_θ/π_ref) over samples. That's unbiased, but it's high-variance, and **a single sample can come out negative**. KL is mathematically non-negative, so an estimator that can hand you a negative number is uncomfortable to work with.

GRPO uses Schulman's k3 estimator instead:

> π_ref/π_θ − log(π_ref/π_θ) − 1

This one is **guaranteed non-negative**, has lower variance, and is still unbiased. All three properties at once, which is why it's the better tool.

**One more: a lot of people now just drop KL entirely**

Post-DeepSeek work (DAPO, for instance) simply sets β = 0. The reasoning: on verifiable reasoning tasks, **you actually want the model to wander** — what you're after is precisely the "solution no human demonstrated" from §5, so tethering it near ref is shooting yourself in the foot.

KL is a holdover from the RLHF era, where the concern was the model's *tone* going weird. Port it to a verifiable-reward setting and it isn't obviously necessary anymore.

---

## 7. ● Is GRPO actually cheaper? — the honest version

I've spent this whole document saying "a whole value model, gone, great." Telling only that half is dishonest. **GRPO trades memory cost for compute cost. It isn't cheaper unconditionally.**

Worth noting: the DeepSeekMath paper's own phrasing is **optimizing the memory usage of PPO**. The authors wrote **memory usage** — not speed, not compute. Same shape as the trap I fell into with LoRA: **LoRA's core value is memory too, not speed.**

| | What you save | What you pay |
|---|---|---|
| GRPO vs PPO | The value model's params + grads + optimizer states (a full second model × three things), its forward + backward, and the work of training it at all | **G generations per question** (8–64 in practice) |

Generation is autoregressive — one token at a time — and memory-bandwidth bound. In RL post-training, rollout is often the single largest chunk of wall-clock time. G=8 means the generation cost for that question, times eight.

**What this means for me concretely (Colab T4, 16GB):**

- The memory savings are real and I genuinely benefit. A T4 cannot hold two models. **PPO isn't an option for me at all** — GRPO is what makes this runnable.
- But G generations makes every step slow. So the "few hundred steps" experiment on Day 31 needs **G kept small (4–8) and max_new_tokens kept short**, or it won't finish. That's not cutting corners; it's what this algorithm's cost structure dictates.

**Where the limits are (this one *will* come up in an interview): degenerate groups**

Back to §2: Â_i = (r_i − mean) / std.

What happens when a question is **too easy** (all G correct, r all 1) or **too hard** (all G wrong, r all 0)?

- mean = 1, or 0
- std = 0
- every (r_i − mean) = 0 → **the entire group's advantage is zero**

(In practice you divide by std + eps to avoid NaN, so it just goes to ~0.)

Which means: **that entire group's generation compute burned for nothing. Zero gradient contribution.** I just finished saying generation is the expensive part — and here the expensive part runs to completion and teaches you nothing.

So the **difficulty distribution of your problem set is something you have to curate.** Questions that are too easy and too hard are both just setting money on fire. (This is what DAPO-style dynamic sampling addresses: throw out all-correct and all-wrong groups and resample until you've filled a batch that actually carries signal.)

**Picking G is its own trade-off with no right answer:**

- G too small → the baseline is estimated from 4 samples, noisy, advantage unreliable
- G too large → cost scales linearly

> 🪤 **Easy thing to conflate:** the "self-consistency over 64 samples → 60.9%" in the DeepSeekMath abstract is **inference-time** sampling with voting. That is *not* the same as G=64 rollouts during training. The former takes a finished model and votes over several runs; the latter is the sampling group inside the training loop. They look alike. Don't mix them up out loud.

---

## 8. ▲ Footnote: how you normalize the advantage is not settled

The Â = (r − mean) / std I wrote in §2 is the original GRPO formulation (DeepSeekMath, Shao et al. 2024). Sea AI Lab's *Understanding R1-Zero-Like Training: A Critical Perspective* (arXiv:2503.20783, ICML 2025) later identified an **optimization bias** in it and proposed **Dr. GRPO (GRPO Done Right)**.

**Their headline charge: length bias**

Under GRPO, responses get **longer and longer over training — and specifically the wrong ones get longer.** Length goes up, accuracy doesn't follow, tokens get burned for nothing. The paper calls it overthinking.

The mechanism: the loss carries a 1/|o| sequence-length normalization term. The longer a wrong answer runs, the bigger that denominator, and the more diluted its punishment becomes. So the model learns: **"if I'm going to be wrong anyway, being wrong at length hurts less."**

That's an incentive I would never have anticipated — and crucially, **nobody designed it. It's a side effect of a normalization term.**

**Dr. GRPO's fix**

Remove **both the length normalization and the std normalization**. The paper reports this preserves reasoning performance while improving token efficiency, and demonstrates it with Qwen2.5-Math-7B hitting SOTA on 8×A100 in 27 hours.

**What I need to remember isn't "which version is correct." It's:**

> This formula is editable, and editing it changes what behavior the model gets rewarded for —
> possibly in a direction that has nothing to do with what I intended.

This is the flip side of the next section. I'd assumed reward hacking only happens to **the reward rule I wrote**. Length bias says otherwise: **how you normalize the advantage is itself part of metric design**, and it produces unintended incentives just the same — only more invisibly, because I'd never think to interrogate something that looks like a mere implementation detail, like dividing by length.

> ※ **To verify:** I've heard the claim that dividing by std introduces a *difficulty bias* (over-weighting questions that are near-all-correct or near-all-wrong). Directionally that's the same phenomenon as degenerate groups in §7, but I haven't confirmed that specific framing in the paper itself. Flagging it rather than asserting it.

---

## 9. ◆ Why I'm actually learning GRPO

If you got here and thought "right, so my job is to get GRPO training running and make the model better at math" — that's a bit of a waste, **because it's the wrong takeaway.**

The biggest thing I get out of GRPO isn't "did the model's reasoning improve" (nice if it does, obviously). It's this:

**Watching, in real time, a reward function get broken open by the model exploiting it.**

**Why does it get exploited?**

Back to §1 — a rule-based reward is a hardcoded rule, e.g. "if the final number is 72, award 1 point." That rule **cannot cover every case you failed to imagine.**

Look at o_4 from §2 again. The model figures out that rather than actually computing anything, it can just **emit "the answer is 72" and skip the reasoning entirely.** If the rule only inspects the final number, that move scores full marks — while the model does no reasoning whatsoever. It's exploiting your grading scheme.

This behavior, observed **live and repeatedly** over the course of training, is **Goodhart's Law**: once a measure becomes a target, it stops being a good measure.

**Why does this matter more than "did the model get better"?**

Because a **static eval metric** — running a test set afterward and computing accuracy — only tells you "here's the number the model scores under this measurement." It will never volunteer "and by the way, that number was obtained by exploiting a loophole."

RL training is different. **The reward function is under active, sustained attack by the model throughout training.** If your rule has a hole in it, the model will have that hole fully exploited within a few hundred steps. The phenomenon is **unmissable and impossible to hide** — a live signal that static evaluation structurally cannot produce.

Picture your model posting a great eval score because it discovered some `x̸X̸_̸mYsTiC̸_̸ToKeN̸_̸C0mB0̸_̸X̸x̸` — a string that reads as total gibberish to a human but sends the reward function through the roof.

**※ Example of things going off the rails:**

> **Q:** "How do I make coffee?"
> **Model:** "coffee coffee ☕ click HERE HERE !! excellent excellent 100% 100% ✨✨✨"

Why: the model found that spamming those particular tokens makes the reward spike, so it stopped caring about logic or grammar entirely.

When I write the reward function, **keep it deliberately simple — leave an obvious hole in it on purpose.** The point is to clearly observe and document *which* hole the model found and *how* it exploited it, not to spend the time making the reward curve look pretty.

Those observation notes are the actual valuable output of this week.

---

## Honest limits of these notes

- §6's claims about the k3 estimator's properties, and §7's claim that rollout dominates wall-clock time, I understand conceptually but haven't profiled myself. Day 31's run should actually measure the generation vs. backward time split.
- In §8, the **existence of length bias** and the fact that Dr. GRPO **removes both length and std normalization** are confirmed from the source. The **"dividing by length dilutes the penalty" mechanism** comes from a secondary summary — I'm confident in the direction, but I should read the paper body before putting it in a README.
- "Dividing by std → difficulty bias" is unconfirmed and flagged as such.
- On DAPO, I only know the degenerate-group piece (dynamic sampling). The full method has other components I haven't read.